# 🌾 AakashaVani (WeatherGPT): 1-Click LLM Fine-Tuning Pipeline
### Domain Adaptation for Indian Meteorology, ICAR Agromet Advisories, & NDMA Disaster Protocols

This notebook trains a **LoRA / QLoRA adapter** on top of **`unsloth/Llama-3.2-1B-Instruct`** or **`google/gemma-2-2b-it`** using a free Google Colab T4 GPU (takes ~10-15 minutes).

In [ ]:
# 1. Install Unsloth & Fast Training Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes datasets transformers

In [ ]:
# 2. Load Base Model with 4-bit Quantization (Fast & Memory Efficient)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/Llama-3.2-1B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. Add LoRA Adapters for Parameter-Efficient Fine-Tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
# 4. Ingest AakashaVani Domain Dataset
from datasets import load_dataset

# Load our curated JSONL training dataset
dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text": texts }

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Loaded {len(dataset)} domain training dialogues!")

In [ ]:
# 5. Train the Model using SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🔥 Starting Training...")
trainer_stats = trainer.train()

In [ ]:
# 6. Test Inference with Fine-Tuned Model
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are AakashaVani, an empathetic conversational AI for weather forecasting, ICAR agromet advisories, and NDMA disaster safety in India."},
    {"role": "user", "content": "Can I spray pesticide on my cotton in Wardha tomorrow? Current humidity is 85% with 20mm rainfall forecast."}
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
# 7. Save and Download Fine-Tuned LoRA Adapter
model.save_pretrained("aakashavani_lora_model")
tokenizer.save_pretrained("aakashavani_lora_model")

!zip -r aakashavani_lora_model.zip aakashavani_lora_model
print("✅ Saved aakashavani_lora_model.zip! You can download it directly from the Colab file explorer.")